# 🚀 Group A — Notebook 3A: Full DNABERT Fine-Tuning on Perlmutter

Notebook 1 taught the model:

```text
DNA → 6-mers → DNABERT → classifier
```

Notebook 3A keeps that same idea and changes the **scale of training**.

```mermaid
flowchart LR
    A["Notebook 1<br/>Understand DNABERT"] --> B["Notebook 3A"]
    B --> C["Full fine-tuning"]
    C --> D["Multiple A100 GPUs"]
    D --> E["BF16"]
    E --> F["DDP"]
    F --> G["Evaluate model + GPU use"]
```

Most Slurm/DDP implementation code stays hidden. Students focus on what resources were requested, what the job does, and how to interpret the result.

# 1. The HPC ideas before the code

## DDP — Distributed Data Parallel

Each GPU contains a copy of the same model but processes different examples.

```mermaid
flowchart TD
    A["One global batch"] --> B["Part 1 → GPU 0 / rank 0"]
    A --> C["Part 2 → GPU 1 / rank 1"]
    A --> D["Part 3 → GPU 2 / rank 2"]
    A --> E["Part 4 → GPU 3 / rank 3"]

    B --> F["Synchronize gradients"]
    C --> F
    D --> F
    E --> F

    F --> G["Same updated DNABERT on every GPU"]
```

### Rank

A **rank** is the ID of one DDP process.

### World size

The total number of DDP processes. With one process per GPU, it is also the number of GPUs.

### BF16

A lower-precision numerical format that lets A100 GPUs perform many training operations more efficiently.

### Local batch vs global batch

```text
local batch  = examples handled by one GPU
global batch = local batch × number of GPUs
```

# 2. The model itself did not change

```mermaid
flowchart LR
    A["DNA"] --> B["6-mers"]
    B --> C["Local DNABERT"]
    C --> D["[CLS] summary"]
    D --> E["Classifier"]
    E --> F["Binding / Background"]
```

Notebook 3A uses the local model:

```text
/global/cfs/cdirs/m4388/projects/project7/models/DNA_bert_6
```

The difference is that **all useful DNABERT parameters are fine-tuned**, and several GPUs cooperate on one training run.

In [ ]:
import os
import sys
import json
import time
import shlex
import subprocess
import py_compile

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# All bootcamp notebooks resolve the SAME folder, so a dataset built in
# Notebook 0 is found by Notebooks 1-3 even if you launch them from
# elsewhere. Override by setting the DNA_BOOTCAMP_HOME environment variable.
PROJECT_DIR = Path(os.environ.get("DNA_BOOTCAMP_HOME", ".")).expanduser().resolve()
print("📁 Project directory:", PROJECT_DIR)

DATA_DIR = Path(
    "/global/cfs/cdirs/m4388/projects/project7/ctcf_k562_example"
)
RESULTS_DIR = (PROJECT_DIR / "notebook3a_results")

SCRIPTS_DIR = (PROJECT_DIR / "notebook3a_scripts")

SLURM_LOG_DIR = (RESULTS_DIR / "slurm_logs")

for directory in [RESULTS_DIR, SCRIPTS_DIR, SLURM_LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# IMPORTANT:
# Do not assume the current Jupyter kernel is the same
# Python environment that should run the Slurm training job.
#
# Test likely dna-llm interpreters and choose the first one
# that can import the packages required by Notebook 3A.
# ---------------------------------------------------------

PYTHON_CANDIDATES = [
    # Shared project environment — best for a bootcamp if available.
    Path("/global/cfs/cdirs/m4388/envs/dna-llm/bin/python"),

    # User-local environment used in earlier Notebook 3 runs.
    Path.home() / ".conda" / "envs" / "dna-llm" / "bin" / "python",

    # Current Jupyter kernel as a final candidate.
    Path(sys.executable)]


def python_environment_check(python_path,):
    """
    Return (works, message).

    A usable training interpreter must import all packages
    needed by the DNABERT Slurm program.
    """

    if not python_path.exists():
        return (False, "path does not exist")

    command = [str(python_path), "-c", ("import sys; " "import torch; "
            "import transformers; " "import sklearn; " "import pandas; "
            "import numpy; " "print(sys.executable); "
            "print('torch=' + torch.__version__); "
            "print('transformers=' + transformers.__version__)")]

    result = subprocess.run(command, capture_output=True, text=True)

    if result.returncode != 0:
        message = (result.stderr.strip() or result.stdout.strip()
            or "import check failed")

        return (False, message)

    return (True, result.stdout.strip())


NOTEBOOK_PYTHON = None

seen = set()

for candidate in PYTHON_CANDIDATES:
    candidate = candidate.resolve()

    if candidate in seen:
        continue

    seen.add(candidate)

    works, message = (python_environment_check(candidate))

    print()
    print("Python candidate:", candidate)

    if works:
        print("✅ Required packages available")

        print(message)

        if NOTEBOOK_PYTHON is None:
            NOTEBOOK_PYTHON = str(candidate)

    else:
        print("❌ Not usable for DNABERT")

        print(message.splitlines()[-1] if message else "unknown error")


if NOTEBOOK_PYTHON is None:
    raise RuntimeError("Notebook 3A could not find a Python interpreter "
        "that imports torch, transformers, sklearn, pandas, "
        "and numpy. Select/repair the dna-llm environment "
        "before submitting the GPU job.")


print()
print("✅ Slurm training Python:", NOTEBOOK_PYTHON)

print("Data directory:", DATA_DIR)

print("Results directory:", RESULTS_DIR)

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)


# 3. Student-controlled settings

The next code cell contains the values you are most likely to change.

### `RUN_MODE`

- `"bootcamp"` → use the bootcamp reservation.
- `"shared"` → small tests outside reservation time.

### `BOOTCAMP_DAY`

Selects the reservation configured for that day.

### `GPU_COUNT`

How many GPUs should cooperate on the job.

### `LOCAL_BATCH_SIZE`

How many DNA examples each GPU processes in one step.

The configuration also contains:

- `epochs` → passes through the training data,
- `learning_rate` → update size,
- `weight_decay` → regularization,
- `warmup_ratio` → gradually increases the learning rate at the start,
- `dropout` → randomly hides some activations during training,
- `precision="bf16"` → use BF16 mixed precision.

In [ ]:
# ✏️ EDIT ME — where you are running, and on how much hardware

NERSC_ACCOUNT = "m4388"

# "shared"   -> testing outside bootcamp hours (1-2 GPUs, no reservation)
# "bootcamp" -> during the bootcamp (uses that day's reservation)
RUN_MODE = "bootcamp"

# Only used when RUN_MODE = "bootcamp". Which day of the bootcamp is it?
BOOTCAMP_DAY = 3

# How many GPUs for the maximum run.
#   shared   : 1 or 2
#   bootcamp : any multiple of 4 (whole nodes), or 1-4 on a single node
GPU_COUNT = 4

# Examples per GPU per step. Raise this if GPU memory stays low.
LOCAL_BATCH_SIZE = 32
GLOBAL_BATCH_SIZE = GPU_COUNT * LOCAL_BATCH_SIZE

MAX_CONFIG = {
    "run_name": "dnabert_maximum_full_finetune",
    "gpus": GPU_COUNT,

    # More training than the short demonstration runs in Notebook 1.
    "epochs": 10,

    # Derived so every GPU gets LOCAL_BATCH_SIZE examples.
    "batch_size": GLOBAL_BATCH_SIZE,

    # Full BERT fine-tuning needs a smaller LR than a randomly
    # initialized classifier head.
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.10,
    "dropout": 0.10,

    # 200 bp -> 195 six-mers + [CLS] + [SEP] = 197 tokens. Padding to
    # 256 would waste 23% of every attention computation on nothing.
    "max_length": 200,

    # A100-oriented precision mode.
    "precision": "bf16",
    "seed": 42,
}

print("Bootcamp day:", BOOTCAMP_DAY)
print("GPUs:", GPU_COUNT)
print("Local batch per GPU:", LOCAL_BATCH_SIZE)
print("Global batch:", GLOBAL_BATCH_SIZE)
print("Epochs:", MAX_CONFIG["epochs"])

In [ ]:
# 🔒 RUN ONLY — turn "I want N GPUs" into Slurm flags that actually get you N

import math

GPUS_PER_NODE = 4          # a Perlmutter GPU node has 4 A100s

# name, nodes held, window
BOOTCAMP_RESERVATIONS = {
    1: ("bootcamp_day1", 20, "11:00-22:00"),
    2: ("bootcamp_day2", 30, "11:00-22:00"),
    3: ("bootcamp_day3", 30, "11:00-22:00"),
    4: ("bootcamp_day4", 40, "08:00-00:00"),
    5: ("bootcamp_day5", 30, "08:00-11:00"),
}


def resolve_slurm(gpu_count, run_mode=None, day=None):
    """Return the Slurm settings needed to obtain `gpu_count` GPUs."""
    run_mode = RUN_MODE if run_mode is None else run_mode
    day = BOOTCAMP_DAY if day is None else day
    gpu_count = int(gpu_count)

    if gpu_count < 1:
        raise ValueError("Request at least one GPU.")

    if run_mode == "shared":
        if gpu_count > 2:
            raise ValueError(
                f"RUN_MODE='shared' is for testing and allows 1-2 GPUs, "
                f"but GPU_COUNT={gpu_count}. During the bootcamp set "
                f"RUN_MODE='bootcamp' and BOOTCAMP_DAY."
            )
        return {"qos": "shared", "reservation": None, "nodes": 1,
                "tasks_per_node": gpu_count, "gpus_per_node": gpu_count,
                "gpus": gpu_count, "window": "any"}

    if run_mode == "bootcamp":
        if day not in BOOTCAMP_RESERVATIONS:
            raise ValueError(f"BOOTCAMP_DAY must be one of "
                             f"{sorted(BOOTCAMP_RESERVATIONS)}, got {day}.")
        name, max_nodes, window = BOOTCAMP_RESERVATIONS[day]
        nodes = math.ceil(gpu_count / GPUS_PER_NODE)

        if nodes > 1 and gpu_count % GPUS_PER_NODE != 0:
            raise ValueError(
                f"A multi-node run must use whole nodes: GPU_COUNT must be a "
                f"multiple of {GPUS_PER_NODE}, got {gpu_count}. Try "
                f"{nodes * GPUS_PER_NODE}."
            )
        if nodes > max_nodes:
            raise ValueError(
                f"{gpu_count} GPUs needs {nodes} nodes, but {name} only holds "
                f"{max_nodes} ({max_nodes * GPUS_PER_NODE} GPUs)."
            )

        per_node = gpu_count if nodes == 1 else GPUS_PER_NODE
        return {"qos": "regular", "reservation": name, "nodes": nodes,
                "tasks_per_node": per_node, "gpus_per_node": per_node,
                "gpus": gpu_count, "window": window}

    raise ValueError("RUN_MODE must be 'shared' or 'bootcamp'.")


PLAN = resolve_slurm(GPU_COUNT)

print("Slurm plan for", GPU_COUNT, "GPU(s)")
print("-" * 42)
for key in ["qos", "reservation", "nodes", "tasks_per_node",
            "gpus_per_node", "window"]:
    print(f"  {key:<15}: {PLAN[key]}")
print()
print(f"  {PLAN['nodes']} node(s) x {PLAN['gpus_per_node']} GPU(s) "
      f"= {PLAN['gpus']} GPU(s) total")
if PLAN["reservation"]:
    print(f"  ⚠️  {PLAN['reservation']} only exists during {PLAN['window']}.")

# 4. Hidden infrastructure

The next hidden cells create:

```text
DDP helper
training program
Slurm job
job monitor
```

You do not need to read hundreds of lines of infrastructure to understand the experiment.

Think of them as building a new function for us:

```python
prepare DNABERT job
submit DNABERT job
```

In [ ]:
%%writefile notebook3a_scripts/ddp_common.py
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist

from sklearn.metrics import (roc_auc_score, average_precision_score,
    confusion_matrix, precision_score, recall_score, f1_score)


def setup_distributed():
    rank = int(os.environ.get("RANK", "0"))

    local_rank = int(os.environ.get("LOCAL_RANK", "0"))

    world_size = int(os.environ.get("WORLD_SIZE", "1"))

    distributed = (world_size > 1)

    visible_gpu_count = (torch.cuda.device_count())

    cuda_visible_devices = (os.environ.get("CUDA_VISIBLE_DEVICES", "<not set>"
        ))

    print(f"[DDP setup] " f"RANK={rank} | " f"LOCAL_RANK={local_rank} | "
        f"WORLD_SIZE={world_size} | " f"visible_GPUs={visible_gpu_count} | "
        f"CUDA_VISIBLE_DEVICES=" f"{cuda_visible_devices}", flush=True)

    if visible_gpu_count < world_size:
        raise RuntimeError(f"Rank {rank} sees only "
            f"{visible_gpu_count} GPU(s), " f"but WORLD_SIZE={world_size}.")

    if local_rank >= visible_gpu_count:
        raise RuntimeError(f"LOCAL_RANK={local_rank}, but only "
            f"{visible_gpu_count} GPU ordinal(s) " "are visible.")

    torch.cuda.set_device(local_rank)

    device = torch.device("cuda", local_rank)

    print(f"[GPU mapping] " f"rank={rank} → cuda:{local_rank} | "
        f"{torch.cuda.get_device_name(local_rank)}", flush=True)

    if distributed:
        dist.init_process_group(
            backend="nccl",
            init_method="env://",
            rank=rank,
            world_size=world_size,
            device_id=device,
        )

    return (distributed, rank, world_size, local_rank, device)


def cleanup_distributed(distributed):
    if (distributed and dist.is_initialized()):
        dist.destroy_process_group()


def reduce_training_stats(loss_sum, correct, n, device, distributed):
    values = torch.tensor([loss_sum, correct, n], dtype=torch.float64,
        device=device)

    if distributed:
        dist.all_reduce(values, op=dist.ReduceOp.SUM)

    loss_total, correct_total, n_total = (values.tolist())

    return (loss_total / n_total, correct_total / n_total, int(n_total))


def binary_metrics(y_true, y_pred, scores):
    tn, fp, fn, tp = (confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel())

    return {"accuracy": ((tp + tn) / (tp + tn + fp + fn)),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "specificity": (tn / (tn + fp) if (tn + fp) else np.nan),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auroc": roc_auc_score(y_true, scores),
        "auprc": average_precision_score(y_true, scores),
        "true_negative": int(tn), "false_positive": int(fp),
        "false_negative": int(fn), "true_positive": int(tp)}

In [ ]:
# 🔒 RUN ONLY — confirm the file landed, and bind it to a variable

# %%writefile above wrote to a path relative to the notebook's working
# directory. SCRIPTS_DIR is the absolute form of that same folder — this
# check fails loudly if the two ever disagree, instead of writing the
# script somewhere the Slurm job will not find it.
DDP_COMMON = SCRIPTS_DIR / "ddp_common.py"

if not DDP_COMMON.exists():
    raise FileNotFoundError(
        f"Expected {DDP_COMMON} after running the %%writefile cell above.\n"
        f"The notebook's working directory is {Path.cwd()}, and %%writefile "
        f"wrote to 'notebook3a_scripts/ddp_common.py' relative to it.\n"
        f"Run the %%writefile cell, or start Jupyter from {PROJECT_DIR}."
    )

py_compile.compile(str(DDP_COMMON), doraise=True)

print("✅", DDP_COMMON)
print(f"   {len(DDP_COMMON.read_text().splitlines()):,} lines, valid Python")

In [ ]:
%%writefile notebook3a_scripts/maximum_dnabert.py
import argparse
import json
import math
import time
from pathlib import Path

# Keep student-facing logs focused on training results.
# Library errors remain visible; routine advisory messages are suppressed.
import warnings

from huggingface_hub import logging as hf_hub_logging
from transformers.utils import logging as transformers_logging

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module=r"torch\.distributed\.c10d_logger",
)

hf_hub_logging.set_verbosity_error()
transformers_logging.set_verbosity_error()

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F

from torch.nn.parallel import DistributedDataParallel as DDP

from torch.utils.data import Dataset, DataLoader

from torch.utils.data.distributed import DistributedSampler

from sklearn.model_selection import train_test_split

from sklearn.metrics import roc_auc_score, average_precision_score

from transformers import (AutoTokenizer, BertModel,
    get_linear_schedule_with_warmup)

from ddp_common import (setup_distributed, cleanup_distributed,
    reduce_training_stats, binary_metrics)


MODEL_NAME = "/global/cfs/cdirs/m4388/projects/project7/models/DNA_bert_6"


def clean_split(data_dir, seed):
    data_dir = Path(data_dir)

    seqs = [line.strip().upper() for line in open(data_dir / "seqs.txt")
        if line.strip()]

    labels = [int(line.strip()) for line in open(data_dir / "labels.txt")
        if line.strip()]

    df = pd.DataFrame({"sequence": seqs, "label": labels})

    df["length"] = (df["sequence"].str.len())

    expected_length = int(df["length"].mode().iloc[0])

    valid = (df["length"].eq(expected_length) & df["sequence"].apply(
            lambda seq: set(seq) <= set("ACGT")))

    conflicts = set(df.groupby("sequence")["label"] .nunique() .loc[
            lambda values: values > 1] .index)

    clean_df = (df[valid & ~df["sequence"].isin(conflicts)] .drop_duplicates(
            "sequence") .reset_index(drop=True))

    train_df, val_df = (train_test_split(clean_df, test_size=0.20,
            random_state=seed, stratify=(clean_df["label"])))

    return (train_df.reset_index(drop=True), val_df.reset_index(drop=True))


def kmer_sentence(seq, k=6):
    return " ".join(seq[i:i+k] for i in range(len(seq) - k + 1))


class DNASet(Dataset):
    def __init__(self, df, tokenizer, max_length):
        sentences = [kmer_sentence(seq) for seq in df["sequence"]]

        encoded = tokenizer(sentences, padding="max_length", truncation=True,
            max_length=max_length, return_tensors="pt")

        self.inputs = (encoded["input_ids"])

        self.masks = (encoded["attention_mask"])

        self.labels = torch.tensor(df["label"].to_numpy(), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {"input": self.inputs[idx], "attention_mask": self.masks[idx],
 "label": self.labels[idx]}


class MaximumDNABertClassifier(nn.Module):
    def __init__(self, dropout):
        super().__init__()

        # IMPORTANT:
        # We remove the BERT pooler entirely because
        # this classifier directly uses the final
        # hidden state of the [CLS] token.
        self.bert = (BertModel.from_pretrained(MODEL_NAME, local_files_only=True,
                add_pooling_layer=False))

        hidden_size = (self.bert.config.hidden_size)

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(hidden_size, 2)

        # FULL FINE-TUNING:
        # every remaining parameter receives gradients.
        for parameter in (self.parameters()):
            parameter.requires_grad = True

    def forward(self, x, attention_mask):
        output = self.bert(input_ids=x, attention_mask=(attention_mask))

        cls_vector = (output .last_hidden_state[:, 0, :])

        return self.classifier(self.dropout(cls_vector))


def parameter_group_table(model,):
    rows = []

    for name, parameter in (model.named_parameters()):
        if name.startswith("bert.embeddings"):
            group = ("BERT embeddings")

        elif ("bert.encoder.layer." in name):
            layer_number = (name.split("bert.encoder.layer.")[1] .split(".")[0]
            )

            group = ("BERT layer " + layer_number)

        elif name.startswith("classifier"):
            group = ("classification head")

        else:
            group = "other"

        rows.append({"parameter": name, "group": group, "numel": (
                parameter.numel()), "trainable": bool(parameter.requires_grad)
        })

    detail = pd.DataFrame(rows)

    grouped = (detail.groupby(["group", "trainable"], as_index=False
        )["numel"] .sum())

    return (detail, grouped)


def build_optimizer(model, learning_rate, weight_decay):
    no_decay_terms = ("bias", "LayerNorm.weight")

    decay_parameters = []
    no_decay_parameters = []

    for name, parameter in (model.named_parameters()):
        if any(term in name for term in no_decay_terms):
            no_decay_parameters.append(parameter)
        else:
            decay_parameters.append(parameter)

    groups = [{"params": decay_parameters,
 "weight_decay": weight_decay}, {"params": no_decay_parameters,
 "weight_decay": 0.0}]

    try:
        optimizer = (torch.optim.AdamW(groups, lr=learning_rate, fused=True))

        optimizer_mode = ("AdamW fused=True")

    except (TypeError, RuntimeError):
        optimizer = (torch.optim.AdamW(groups, lr=learning_rate))

        optimizer_mode = ("AdamW standard")

    return (optimizer, optimizer_mode)


def train_epoch(model, optimizer, scheduler, loader, sampler, epoch, device,
    distributed):
    model.train()

    if sampler is not None:
        sampler.set_epoch(epoch)

    loss_sum = 0.0
    correct = 0
    n = 0

    for batch_index, batch in enumerate(loader):
        x = batch["input"].to(device, non_blocking=True)

        mask = batch["attention_mask"].to(device, non_blocking=True)

        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(x, mask)

            loss = (F.cross_entropy(logits, labels))

        loss.backward()

        # Full-fine-tuning audit:
        # on the first batch every trainable parameter
        # should have a gradient.
        if (epoch == 1 and batch_index == 0):
            missing_grads = [name for name, parameter
                in model.named_parameters() if (parameter.requires_grad
                    and parameter.grad is None)]

            if missing_grads:
                raise RuntimeError("Trainable parameters "
                    "without gradients: " + ", ".join(missing_grads))

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        predictions = (logits.argmax(dim=1))

        loss_sum += (loss.item() * len(labels))

        correct += (predictions == labels).sum().item()

        n += len(labels)

    return reduce_training_stats(loss_sum, correct, n, device, distributed)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()

    loss_sum = 0.0
    correct = 0
    n = 0

    scores = []
    predictions = []
    truth = []

    for batch in loader:
        x = batch["input"].to(device, non_blocking=True)

        mask = batch["attention_mask"].to(device, non_blocking=True)

        labels = batch["label"].to(device, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(x, mask)

            loss = (F.cross_entropy(logits, labels))

        pred = logits.argmax(dim=1)

        prob = torch.softmax(logits.float(), dim=1)[:, 1]

        loss_sum += (loss.item() * len(labels))

        correct += (pred == labels).sum().item()

        n += len(labels)

        scores.extend(prob.cpu().numpy())

        predictions.extend(pred.cpu().numpy())

        truth.extend(labels.cpu().numpy())

    return {"loss": loss_sum / n, "accuracy": correct / n,
 "scores": np.asarray(scores), "predictions": np.asarray(predictions),
 "true": np.asarray(truth)}


def main():
    parser = (argparse.ArgumentParser())

    parser.add_argument("--data_dir", required=True)

    parser.add_argument("--output_dir", required=True)

    parser.add_argument("--run_name", required=True)

    parser.add_argument("--epochs", type=int, required=True)

    parser.add_argument("--batch_size", type=int, required=True)

    parser.add_argument("--learning_rate", type=float, required=True)

    parser.add_argument("--weight_decay", type=float, required=True)

    parser.add_argument("--warmup_ratio", type=float, required=True)

    parser.add_argument("--dropout", type=float, required=True)

    # 200 bp -> 195 six-mers + [CLS] + [SEP] = 197 tokens.
    parser.add_argument("--max_length", type=int, default=200)

    parser.add_argument("--precision", choices=["bf16",], default="bf16")

    parser.add_argument("--seed", type=int, default=42)

    args = (parser.parse_args())

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU required.")

    if not (torch.cuda.is_bf16_supported()):
        raise RuntimeError("Notebook 3A expects " "BF16-capable GPUs.")

    (distributed, rank, world_size, local_rank, device) = setup_distributed()

    try:
        if (args.batch_size % world_size != 0):
            raise ValueError("Global batch size " "must be divisible "
                "by GPU count.")

        local_batch_size = (args.batch_size // world_size)

        np.random.seed(args.seed)

        torch.manual_seed(args.seed)

        torch.cuda.manual_seed_all(args.seed)

        # Allow optimized matmul paths for remaining FP32 ops.
        torch.set_float32_matmul_precision("high")

        train_df, val_df = (clean_split(args.data_dir, args.seed))

        tokenizer = (AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True))

        train_dataset = DNASet(train_df, tokenizer, args.max_length)

        val_dataset = DNASet(val_df, tokenizer, args.max_length)

        train_sampler = None

        if distributed:
            train_sampler = (DistributedSampler(train_dataset, num_replicas=(
                        world_size), rank=rank, shuffle=True, seed=args.seed))

        loader_workers = min(4, max(1, (int(__import__("os").environ.get(
                            "SLURM_CPUS_PER_TASK", "4")) // 4)))

        train_loader = (DataLoader(train_dataset, batch_size=(local_batch_size
                ), shuffle=(train_sampler is None), sampler=(train_sampler),
                pin_memory=True, num_workers=(loader_workers),
                persistent_workers=(loader_workers > 0)))

        # Rank 0 performs validation.
        val_loader = (DataLoader(val_dataset, batch_size=(local_batch_size),
                shuffle=False, pin_memory=True, num_workers=(loader_workers),
                persistent_workers=(loader_workers > 0)))

        model = (MaximumDNABertClassifier(dropout=(args.dropout)) .to(device))

        total_parameters = sum(p.numel() for p in model.parameters())

        trainable_parameters = sum(p.numel() for p in model.parameters()
            if p.requires_grad)

        trainable_percent = (100.0 * trainable_parameters / total_parameters)

        if (trainable_parameters != total_parameters):
            raise RuntimeError("Maximum fine-tuning " "requires all remaining "
                "model parameters to " "be trainable.")

        parameter_detail, (parameter_groups) = parameter_group_table(model)

        if distributed:
            model = DDP(model, device_ids=[device.index], output_device=(
                    device.index), gradient_as_bucket_view=True,
                static_graph=True)

        (optimizer, optimizer_mode) = build_optimizer(model,
            args.learning_rate, args.weight_decay)

        total_steps = (len(train_loader) * args.epochs)

        warmup_steps = int(total_steps * args.warmup_ratio)

        scheduler = (get_linear_schedule_with_warmup(optimizer,
                num_warmup_steps=(warmup_steps), num_training_steps=(
                    total_steps)))

        # Different dropout streams after model sync.
        torch.manual_seed(args.seed + rank)

        torch.cuda.manual_seed_all(args.seed + rank)

        if rank == 0:
            print("\n=== MAXIMUM DNABERT ===", flush=True)

            print(f"GPUs: {world_size}", flush=True)

            print(f"Precision: " f"{args.precision}", flush=True)

            print(f"Global batch: " f"{args.batch_size}", flush=True)

            print(f"Local batch/GPU: " f"{local_batch_size}", flush=True)

            print(f"Trainable parameters: " f"{trainable_parameters:,}",
                flush=True)

            print(f"Total parameters: " f"{total_parameters:,}", flush=True)

            print(f"Trainable percent: " f"{trainable_percent:.2f}%",
                flush=True)

            print(f"Optimizer: " f"{optimizer_mode}", flush=True)

            print(f"Steps/epoch/rank: " f"{len(train_loader)}", flush=True)

        if distributed:
            dist.barrier()

        torch.cuda.empty_cache()

        torch.cuda.reset_peak_memory_stats(device)

        torch.cuda.synchronize(device)

        training_start = (time.time())

        history = []
        final_val = None
        processed_examples = 0

        # Keep the BEST epoch, not the last one. A 10-epoch run has plenty of
        # chances to end on a worse epoch than its peak, and reporting the
        # last one understates the model you actually trained.
        best_auroc = -1.0
        best_val = None
        best_epoch = 0
        best_state = None

        for epoch in range(1, args.epochs + 1):
            epoch_start = (time.time())

            (train_loss, train_accuracy, global_examples_seen) = train_epoch(
                model, optimizer, scheduler, train_loader, train_sampler,
                epoch, device, distributed)

            processed_examples += (global_examples_seen)

            if distributed:
                dist.barrier()

            if rank == 0:
                eval_model = (model.module if distributed else model)

                final_val = evaluate(eval_model, val_loader, device)

                val_auroc = (roc_auc_score(final_val["true"], final_val[
                            "scores"]))

                val_auprc = (average_precision_score(final_val["true"],
                        final_val["scores"]))

                epoch_seconds = (time.time() - epoch_start)

                row = {"epoch": epoch, "train_loss": train_loss,
 "train_accuracy": train_accuracy, "val_loss": final_val["loss"],
 "val_accuracy": final_val["accuracy"], "val_auroc": val_auroc,
 "val_auprc": val_auprc, "epoch_seconds": epoch_seconds,
 "learning_rate": optimizer.param_groups[0]["lr"]}

                history.append(row)

                if val_auroc > best_auroc:
                    best_auroc = val_auroc
                    best_val = final_val
                    best_epoch = epoch
                    best_state = {k: v.detach().cpu().clone()
                                  for k, v in eval_model.state_dict().items()}

                if final_val["loss"] > 0.69 and val_auroc < 0.55:
                    print("  \U0001F6A8 Collapsed to chance "
                          "(val loss near ln(2)). Lower --learning_rate.",
                          flush=True)

                print(f"Epoch " f"{epoch}/" f"{args.epochs} | " f"train loss="
                    f"{train_loss:.4f} | " f"train acc="
                    f"{train_accuracy:.3f} | " f"val loss="
                    f"{final_val['loss']:.4f} | " f"AUROC="
                    f"{val_auroc:.4f} | " f"AUPRC=" f"{val_auprc:.4f} | "
                    f"{epoch_seconds:.1f}s", flush=True)

            if distributed:
                dist.barrier()

        torch.cuda.synchronize(device)

        training_time = (time.time() - training_start)

        local_peak_bytes = (torch.cuda .max_memory_allocated(device))

        peak_tensor = torch.tensor([float(local_peak_bytes)],
            dtype=torch.float64, device=device)

        if distributed:
            dist.all_reduce(peak_tensor, op=dist.ReduceOp.MAX)

        max_peak_bytes = (peak_tensor.item())

        total_memory_bytes = (torch.cuda .get_device_properties(device)
            .total_memory)

        peak_memory_gb = (max_peak_bytes / (1024 ** 3))

        total_memory_gb = (total_memory_bytes / (1024 ** 3))

        peak_memory_percent = (100.0 * max_peak_bytes / total_memory_bytes)

        if rank == 0:
            history_df = (pd.DataFrame(history))

            best_row = (history_df.loc[history_df["val_auroc"].idxmax()])

            # Restore the best epoch before reporting, checkpointing or
            # writing predictions, so everything below describes the model
            # that was actually kept.
            eval_model = (model.module if distributed else model)

            if best_state is not None:
                eval_model.load_state_dict(best_state)
                final_val = best_val
                print(f"\u21A9\uFE0F  Restored weights from epoch "
                      f"{best_epoch} (AUROC {best_auroc:.4f}).", flush=True)

            metrics = (binary_metrics(final_val["true"], final_val[
                        "predictions"], final_val["scores"]))

            predictions_df = (val_df[["sequence", "label"]] .copy())

            predictions_df["predicted_label"] = (final_val["predictions"])

            predictions_df["binding_probability"] = (final_val["scores"])

            predictions_df["correct"] = (predictions_df["label"]
                == predictions_df["predicted_label"])

            # Use the actual number of examples processed by
            # all DDP ranks. DistributedSampler can pad a small
            # number of samples to make rank lengths equal.
            examples_per_second = (processed_examples / training_time)

            output_dir = Path(args.output_dir)

            output_dir.mkdir(parents=True, exist_ok=True)

            prefix = (output_dir / args.run_name)

            history_df.to_csv(str(prefix) + "_history.csv", index=False)

            predictions_df.to_csv(str(prefix) + "_predictions.csv",
                index=False)

            parameter_detail.to_csv(str(prefix) + "_parameters.csv",
                index=False)

            parameter_groups.to_csv(str(prefix) + "_parameter_groups.csv",
                index=False)

            eval_model = (model.module if distributed else model)

            # Save after the training timer so checkpoint I/O
            # does not distort the reported training time.
            checkpoint_path = (str(prefix) + "_final_checkpoint.pt")

            torch.save({"model_name": MODEL_NAME,
 "state_dict": eval_model.state_dict(), "epoch": int(best_epoch),
 "max_length": args.max_length, "dropout": args.dropout}, checkpoint_path)

            summary = {"family": "dnabert", "run_name": args.run_name,
 "model_name": MODEL_NAME, "strategy": "maximum_full_finetuning",
 "distributed": bool(distributed), "num_gpus": int(world_size),
 "precision": args.precision, "epochs": int(args.epochs),
 "global_batch_size": int(args.batch_size),
 "local_batch_size": int(local_batch_size),
 "learning_rate": float(args.learning_rate),
 "weight_decay": float(args.weight_decay),
 "warmup_ratio": float(args.warmup_ratio), "dropout": float(args.dropout),
 "max_length": int(args.max_length), "random_seed": int(args.seed),
 "optimizer": optimizer_mode,
 "trainable_parameters": int(trainable_parameters),
 "total_parameters": int(total_parameters),
 "trainable_percent": float(trainable_percent),
 "best_val_auroc": float(best_row["val_auroc"]),
 "best_epoch": int(best_row["epoch"]),
 "final_val_accuracy": float(metrics["accuracy"]),
 "final_val_precision": float(metrics["precision"]),
 "final_val_recall": float(metrics["recall"]),
 "final_val_specificity": float(metrics["specificity"]),
 "final_val_f1": float(metrics["f1"]),
 "final_val_auroc": float(metrics["auroc"]),
 "final_val_auprc": float(metrics["auprc"]),
 "training_time_seconds": float(training_time),
 "examples_per_second": float(examples_per_second),
 "peak_gpu_memory_gb": float(peak_memory_gb),
 "gpu_memory_capacity_gb": float(total_memory_gb),
 "peak_gpu_memory_percent": float(peak_memory_percent),
 "checkpoint_path": checkpoint_path}

            with open(str(prefix) + "_summary.json", "w") as handle:
                json.dump(summary, handle, indent=2)

            print()
            print("=== FINAL REPORT ===", flush=True)

            print(f"Best AUROC: " f"{summary['best_val_auroc']:.4f}",
                flush=True)

            print(f"Training time: " f"{training_time:.1f}s", flush=True)

            print(f"Examples/sec: " f"{examples_per_second:.1f}", flush=True)

            print(f"Peak GPU memory: " f"{peak_memory_gb:.2f} / "
                f"{total_memory_gb:.2f} GB " f"({peak_memory_percent:.1f}%)",
                flush=True)

            print(f"Checkpoint: " f"{checkpoint_path}", flush=True)

    finally:
        cleanup_distributed(distributed)


if __name__ == "__main__":
    main()

In [ ]:
# 🔒 RUN ONLY — confirm the file landed, and bind it to a variable

# %%writefile above wrote to a path relative to the notebook's working
# directory. SCRIPTS_DIR is the absolute form of that same folder — this
# check fails loudly if the two ever disagree, instead of writing the
# script somewhere the Slurm job will not find it.
MAX_SCRIPT = SCRIPTS_DIR / "maximum_dnabert.py"

if not MAX_SCRIPT.exists():
    raise FileNotFoundError(
        f"Expected {MAX_SCRIPT} after running the %%writefile cell above.\n"
        f"The notebook's working directory is {Path.cwd()}, and %%writefile "
        f"wrote to 'notebook3a_scripts/maximum_dnabert.py' relative to it.\n"
        f"Run the %%writefile cell, or start Jupyter from {PROJECT_DIR}."
    )

py_compile.compile(str(MAX_SCRIPT), doraise=True)

print("✅", MAX_SCRIPT)
print(f"   {len(MAX_SCRIPT.read_text().splitlines()):,} lines, valid Python")

In [ ]:
# 🔒 RUN ONLY — syntax validation + submission monitoring

def validate_shell_script(path,):
    result = subprocess.run(["bash", "-n", str(path)], capture_output=True,
        text=True)

    if result.returncode != 0:
        print(result.stderr)
        return False

    print("✅ Shell syntax valid:", path.name)

    return True


def submit_and_stream(script_path, job_name, poll_seconds=2.0):
    # NERSC Jupyter may itself have a CUDA_VISIBLE_DEVICES
    # value. Do not pass that mask into a new batch job.
    submit_env = (os.environ.copy())

    inherited_gpu_env = {name: submit_env.get(name)
 for name in ["CUDA_VISIBLE_DEVICES", "NVIDIA_VISIBLE_DEVICES",
            "ROCR_VISIBLE_DEVICES", "GPU_DEVICE_ORDINAL"]
 if name in submit_env}

    for name in ["CUDA_VISIBLE_DEVICES", "NVIDIA_VISIBLE_DEVICES",
        "ROCR_VISIBLE_DEVICES", "GPU_DEVICE_ORDINAL"]:
        submit_env.pop(name, None)

    print("Notebook GPU environment:", inherited_gpu_env if inherited_gpu_env
        else "<none>")

    print("Submitting with inherited GPU " "visibility removed.")

    submit = subprocess.run(["sbatch", str(script_path)], capture_output=True,
        text=True, env=submit_env)

    if submit.returncode != 0:
        print(submit.stderr)

        raise RuntimeError("sbatch rejected the job.")

    submit_text = (submit.stdout.strip())

    job_id = (submit_text.split()[-1])

    output_file = (SLURM_LOG_DIR / f"{job_name}-{job_id}.out")

    print("Submitted job", job_id)

    print("Output:", output_file)

    last_size = 0

    while True:
        if output_file.exists():
            with output_file.open("r") as handle:
                handle.seek(last_size)

                text = handle.read()

                if text:
                    print(text, end="")

                last_size = (handle.tell())

        active = subprocess.run(["squeue", "-h", "-j", job_id],
            capture_output=True, text=True).stdout.strip()

        if not active:
            if output_file.exists():
                with output_file.open("r") as handle:
                    handle.seek(last_size)

                    text = (handle.read())

                    if text:
                        print(text, end="")

            break

        time.sleep(poll_seconds)

    summary = subprocess.run(["sacct", "-j", job_id, "--format="
            "JobID,State,ExitCode," "Elapsed,AllocTRES", "-n", "-P"],
        capture_output=True, text=True).stdout.strip()

    print("\n--- sacct summary ---")

    print(summary)

    main_state = None

    for line in (summary.splitlines()):
        fields = (line.split("|"))

        if (len(fields) >= 2 and fields[0] == job_id):
            main_state = (fields[1])
            break

    if (main_state is None or not main_state.startswith("COMPLETED")):
        raise RuntimeError(f"SLURM job {job_id} " f"finished with state "
            f"{main_state}.")

    print(f"✅ Job {job_id} " "completed successfully.")

    return job_id

In [ ]:
# 🔒 RUN ONLY — verify the selected Slurm Python one more time

preflight = subprocess.run([NOTEBOOK_PYTHON, "-c", ("import sys; "
            "import torch; " "import transformers; "
            "print('python:', sys.executable); "
            "print('torch:', torch.__version__); "
            "print('transformers:', transformers.__version__); "
            "print('CUDA build:', torch.version.cuda)")], capture_output=True,
    text=True)

print(preflight.stdout)

if preflight.returncode != 0:
    print(preflight.stderr)

    raise RuntimeError("Selected Slurm Python failed the PyTorch preflight.")

print("✅ Python environment preflight passed")

In [ ]:
# 🔒 RUN ONLY — build command-line arguments

def build_max_arguments(config,):
    values = ["--data_dir", str(DATA_DIR), "--output_dir", str(RESULTS_DIR),
 "--run_name", str(config["run_name"]), "--epochs", str(config["epochs"]),
 "--batch_size", str(config["batch_size"]),
 "--learning_rate", str(config["learning_rate"]),
 "--weight_decay", str(config["weight_decay"]),
 "--warmup_ratio", str(config["warmup_ratio"]),
 "--dropout", str(config["dropout"]),
 "--max_length", str(config["max_length"]),
 "--precision", str(config["precision"]), "--seed", str(config["seed"])]

    return " ".join(shlex.quote(value) for value in values)

In [ ]:
# 🔒 RUN ONLY — generate one DDP Slurm job

def write_max_slurm_job(config, walltime="00:45:00", run_name=None):
    """Turn a config dict into a submittable .slurm file."""
    config = dict(config)
    if run_name is not None:
        config["run_name"] = run_name

    gpu_count = int(config["gpus"])
    plan = resolve_slurm(gpu_count)

    if config["batch_size"] % gpu_count != 0:
        raise ValueError(
            f"Global batch {config['batch_size']} is not divisible by "
            f"{gpu_count} GPUs — every GPU must get the same number of "
            f"examples."
        )

    arguments = build_max_arguments(config)
    job_name = ("nb3a-" + config["run_name"].replace("_", "-"))[:60]
    path = SCRIPTS_DIR / (config["run_name"] + ".slurm")

    # The #SBATCH header is built as a list so the reservation line is
    # plainly present or absent, instead of buried inside an f-string.
    header = [
        "#!/bin/bash",
        f"#SBATCH -A {NERSC_ACCOUNT}",
        "#SBATCH -C gpu",
        f"#SBATCH -q {plan['qos']}",
        f"#SBATCH -t {walltime}",
        "",
        f"#SBATCH -N {plan['nodes']}",
        f"#SBATCH --ntasks-per-node={plan['tasks_per_node']}",
        "#SBATCH --cpus-per-task=32",
        f"#SBATCH --gpus-per-node={plan['gpus_per_node']}",
        "#SBATCH --gpu-bind=none",
        "",
        f"#SBATCH -J {job_name}",
        f"#SBATCH -o {SLURM_LOG_DIR}/{job_name}-%j.out",
    ]
    if plan["reservation"]:
        header.insert(4, f"#SBATCH --reservation={plan['reservation']}")

    body = f"""
export SLURM_CPU_BIND="cores"

export MASTER_ADDR=$(scontrol show hostnames "$SLURM_JOB_NODELIST" | head -n 1)
export MASTER_PORT=$((10000 + SLURM_JOB_ID % 50000))

export NCCL_DEBUG=VERSION

echo "[PYTHON] training interpreter: {NOTEBOOK_PYTHON}"

{NOTEBOOK_PYTHON} -c "import sys, torch, transformers; print('[PYTHON]', sys.executable); print('[PYTORCH]', torch.__version__); print('[TRANSFORMERS]', transformers.__version__)"

if [ $? -ne 0 ]; then
    echo "[ERROR] Selected Python cannot import the DNABERT dependencies."
    exit 1
fi

echo "[BATCH] CUDA_VISIBLE_DEVICES before cleanup=${{CUDA_VISIBLE_DEVICES-<unset>}}"
echo "[BATCH] SLURM_JOB_GPUS=${{SLURM_JOB_GPUS-<unset>}}"

# Prevent the Jupyter server's GPU mask from restricting this job.
unset CUDA_VISIBLE_DEVICES
unset NVIDIA_VISIBLE_DEVICES
unset ROCR_VISIBLE_DEVICES
unset GPU_DEVICE_ORDINAL

srun --gpu-bind=none bash -c '
    export RANK=$SLURM_PROCID
    export LOCAL_RANK=$SLURM_LOCALID
    export WORLD_SIZE=$SLURM_NTASKS

    echo "[SLURM→DDP] RANK=$RANK LOCAL_RANK=$LOCAL_RANK WORLD_SIZE=$WORLD_SIZE CUDA_VISIBLE_DEVICES=${{CUDA_VISIBLE_DEVICES-<unset>}}"

    {NOTEBOOK_PYTHON} {MAX_SCRIPT} {arguments}
'
"""

    path.write_text("\n".join(header) + "\n" + body)

    if not validate_shell_script(path):
        raise RuntimeError("Fix shell syntax before submitting.")

    return path, job_name

# 4. From notebook cell to Perlmutter job

```mermaid
flowchart LR
    A["Student settings"] --> B["write_max_slurm_job(...)"]
    B --> C[".slurm file"]
    C --> D["submit_and_stream(...)"]
    D --> E["Slurm allocates GPUs"]
    E --> F["One DDP rank per GPU"]
    F --> G["Full DNABERT training"]
    G --> H["CSV / JSON results"]
```

## Function: `write_max_slurm_job(...)`

This helper converts the settings into Slurm instructions.

You do **not** need to memorize `#SBATCH` syntax.

## Function: `submit_and_stream(...)`

This submits the generated job and prints useful output while it runs.

# 5. Prepare the job

This cell does **not** train yet.

It creates the Slurm instructions Perlmutter needs.

In [ ]:
# ▶️ RUN — prepare one multi-GPU DNABERT job

MAX_SLURM_SCRIPT, MAX_JOB_NAME = write_max_slurm_job(
    MAX_CONFIG,
    walltime="00:45:00",
)

print("Job is ready.")
print("GPUs:", GPU_COUNT)
print("Run name:", MAX_CONFIG["run_name"])
print("Slurm file:", MAX_SLURM_SCRIPT)

# 6. Submit the job

This is the only line that actually sends the work to Slurm:

```python
submit_and_stream(...)
```

While it runs, focus on the useful output:

```text
rank → GPU mapping
epoch
loss
AUROC
training time
examples / second
peak GPU memory
```

In [ ]:
# ▶️ RUN — start the training job

MAX_JOB_ID = submit_and_stream(
    MAX_SLURM_SCRIPT,
    MAX_JOB_NAME,
)

# 7. Evaluate the result

The training program saves a small JSON summary.

We load that summary so students do not need to inspect a large log.

In [ ]:
# 🔒 HELPER — load result files created by the Slurm job

RUN_NAME = MAX_CONFIG["run_name"]

SUMMARY_PATH = RESULTS_DIR / f"{RUN_NAME}_summary.json"
HISTORY_PATH = RESULTS_DIR / f"{RUN_NAME}_history.csv"
PREDICTIONS_PATH = RESULTS_DIR / f"{RUN_NAME}_predictions.csv"

if not SUMMARY_PATH.exists():
    raise FileNotFoundError(
        "Training summary not found. Make sure the Slurm job completed."
    )

with open(SUMMARY_PATH) as handle:
    max_summary = json.load(handle)

max_history = pd.read_csv(
    HISTORY_PATH
)

max_predictions = pd.read_csv(
    PREDICTIONS_PATH
)

In [ ]:
# ▶️ RUN — important scientific + HPC results

result_table = pd.DataFrame([{
    "GPUs": max_summary["num_gpus"],
    "Trainable %": max_summary["trainable_percent"],
    "Best AUROC": max_summary["best_val_auroc"],
    "Training seconds": max_summary["training_time_seconds"],
    "Examples / second": max_summary["examples_per_second"],
    "Peak GPU memory (GB)": max_summary["peak_gpu_memory_gb"],
}])

result_table.round(3)

In [ ]:
# ▶️ RUN — learning curves

plt.plot(
    max_history["epoch"],
    max_history["train_loss"],
    marker="o",
    label="Training loss",
)

plt.plot(
    max_history["epoch"],
    max_history["val_loss"],
    marker="o",
    label="Validation loss",
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Full DNABERT fine-tuning: loss")
plt.legend()
plt.show()


plt.plot(
    max_history["epoch"],
    max_history["val_auroc"],
    marker="o",
)

plt.xlabel("Epoch")
plt.ylabel("Validation AUROC")
plt.ylim(0, 1)
plt.title("Full DNABERT fine-tuning: AUROC")
plt.show()

## Graph 1 — Learning curves

The loss curves answer:

> Did optimization improve the model, and did validation follow training?

The AUROC-by-epoch graph answers:

> At which epoch did class separation look strongest?

## Graph 2 — Confusion matrix

The predictions file contains:

- `label` → true answer,
- `predicted_label` → model's chosen class,
- `binding_probability` → confidence for the Binding class.

The confusion matrix shows which class mistakes were made.

In [ ]:
cm = confusion_matrix(
    max_predictions["label"],
    max_predictions["predicted_label"],
)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Background", "Binding"],
).plot()

plt.title("Full DNABERT confusion matrix")
plt.show()

## Graph 3 — ROC curve

This evaluates ranking across many possible probability thresholds.

In [ ]:
fpr, tpr, _ = roc_curve(
    max_predictions["label"],
    max_predictions["binding_probability"],
)

plt.plot(
    fpr,
    tpr,
    label=f"AUROC = {max_summary['final_val_auroc']:.3f}",
)

plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Full DNABERT ROC curve")
plt.legend()
plt.show()

## Graph 4 — Precision–Recall curve

This focuses on the tradeoff between finding binding sequences and keeping positive predictions trustworthy.

In [ ]:
precision, recall, _ = precision_recall_curve(
    max_predictions["label"],
    max_predictions["binding_probability"],
)

plt.plot(
    recall,
    precision,
    label=f"AUPRC = {max_summary['final_val_auprc']:.3f}",
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Full DNABERT Precision–Recall curve")
plt.legend()
plt.show()

## Graph 5 — GPU memory use

Peak memory is **not the same as compute utilization**, but it helps answer:

> Did our batch leave lots of GPU memory unused, or are we close to the memory limit?

In [ ]:
memory_percent = max_summary["peak_gpu_memory_percent"]

plt.bar(
    ["Peak allocated GPU memory"],
    [memory_percent],
)

plt.axhline(
    100,
    linestyle="--",
)

plt.ylabel("% of GPU memory capacity")
plt.ylim(0, 105)
plt.title("How much GPU memory did this run use?")
plt.show()

# ✅ Group A final questions

1. What is the difference between **local batch** and **global batch**?
2. What does one DDP rank do?
3. Why do the GPUs synchronize gradients?
4. Did training loss and validation loss move together?
5. What kinds of errors appear in the confusion matrix?
6. What do AUROC and AUPRC tell you that accuracy does not?
7. How much GPU memory did the run use?
8. If GPU memory were very low, what setting could you consider increasing?

# 🎨 Optional Visualization Playground

This section is **optional**. Nothing below is required to finish the notebook.

Use it when you want to ask your own question about a variable you created earlier.

```mermaid
flowchart LR
    A["Choose a variable"] --> B["Choose a term / column"]
    B --> C{"What do you want to see?"}
    C -->|"Counts / categories"| D["Bar plot"]
    C -->|"Distribution"| E["Histogram"]
    C -->|"Change across epochs"| F["Line plot"]
    C -->|"Relationship between numbers"| G["Scatter plot"]
```

## Two plotting patterns to remember

### One term / column

```python
VARIABLE["TERM"].plot(kind="hist")
```

Read it as:

> From this variable, choose this term, then plot it.

### Two terms / columns

```python
VARIABLE.plot(
    x="TERM1",
    y="TERM2",
    kind="scatter",
)
```

Read it as:

> Use `TERM1` for the x-axis and `TERM2` for the y-axis.

Useful `kind=` choices:

| `kind` | Good for |
|---|---|
| `"bar"` | comparing categories |
| `"hist"` | seeing a distribution |
| `"line"` | following change across epochs |
| `"scatter"` | comparing two numerical values |
| `"box"` | comparing distributions between groups |

If you forget what terms exist inside a pandas table, run:

```python
VARIABLE.columns.tolist()
```

## Important variables from Notebook 3A

| Variable | What it contains | Terms you can explore |
|---|---|---|
| `max_history` | one row per DNABERT training epoch | `epoch`, `train_loss`, `train_accuracy`, `val_loss`, `val_accuracy`, `val_auroc`, `val_auprc`, `epoch_seconds` |
| `max_predictions` | validation DNA and predictions | `sequence`, `label`, `predicted_label`, `binding_probability`, `correct` |
| `summary_df` | final scientific + HPC values from the run | examples include `num_gpus`, `global_batch_size`, `local_batch_size`, `final_val_auroc`, `final_val_auprc`, `training_time_seconds`, `examples_per_second`, `peak_gpu_memory_gb`, `peak_gpu_memory_percent` |
| `result_table` | a smaller summary shown earlier | `GPUs`, `Trainable %`, `Best AUROC`, `Training seconds`, `Examples / second`, `Peak GPU memory (GB)` |

This notebook contains **two kinds of values**:

```text
Scientific values
→ loss, accuracy, AUROC, AUPRC, prediction probability

HPC values
→ GPUs, time, examples/second, GPU memory
```

In [ ]:
# ▶️ OPTIONAL — make the summary dictionary easy to inspect with pandas

summary_df = pd.DataFrame([max_summary])

print("max_history terms:")
print(max_history.columns.tolist())

print("\nmax_predictions terms:")
print(max_predictions.columns.tolist())

print("\nsummary_df terms:")
print(summary_df.columns.tolist())

## Copy a visualization recipe and change the terms

### Training loss

```python
max_history.plot(
    x="epoch",
    y=["train_loss", "val_loss"],
    kind="line",
    marker="o",
)
```

### Accuracy across epochs

```python
max_history.plot(
    x="epoch",
    y=["train_accuracy", "val_accuracy"],
    kind="line",
    marker="o",
)
```

### AUROC and AUPRC across epochs

```python
max_history.plot(
    x="epoch",
    y=["val_auroc", "val_auprc"],
    kind="line",
    marker="o",
)
```

### Time per epoch

```python
max_history.plot(
    x="epoch",
    y="epoch_seconds",
    kind="bar",
)
```

### Prediction confidence

```python
max_predictions["binding_probability"].plot(
    kind="hist",
    bins=20,
)
```

### Correct vs. incorrect predictions

```python
max_predictions["correct"].value_counts().plot(kind="bar")
```

### HPC summary values

```python
summary_df[
    [
        "examples_per_second",
        "peak_gpu_memory_gb",
    ]
].T.plot(kind="bar", legend=False)
```

**Important:** values with different units should usually be interpreted separately. For example, `examples_per_second` and `peak_gpu_memory_gb` are useful to inspect, but they do not measure the same thing.